# Stage 9, observe: an operations agent over Airflow and the Spark History Server

Northwind Gear's daily pipeline runs as the Airflow DAG **`northwind_daily`** (`ops/dags/northwind_daily.py`, stage 8):
it checks its inputs, MERGEs the raw orders into `shop_bronze.orders_current`, runs data-quality checks and publishes
`shop_gold.product_kpis`. The Spark notebooks of stages 1–4 leave their applications in the Spark History Server.

Instead of clicking through two UIs, this notebook gives a local LLM the tools of two MCP servers the chart runs:

* **Airflow MCP** (Astronomer `astro-airflow-mcp`, read-only, logged in as its own `viewer` user): DAGs, runs, task
  instances, logs.
* **Spark History MCP** (Kubeflow `mcp-apache-spark-history-server`): applications, jobs, stages, executors.

Helm chart only (`airflowMcp.enabled`, `sparkHistoryMcp.enabled`; the workstation profile turns both on, and mounts
`ops/dags` as Airflow's DAG folder, so the DAG is already deployed and unpaused). On Docker Compose the first cell
stops with a message.

In [ ]:
import sys, pathlib, json
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import pandas as pd
import lab_utils as lu

lu.require_ops()
lu.require_tool_model()
for server, url in lu.ops_servers().items():
    names = sorted(t["name"] for t in lu.ops_tools(server))
    print(f"{server} ({url}): {len(names)} tools\n   {', '.join(names)}\n")

## 1. One tool call by hand
Every MCP tool is a function with a JSON schema. Before handing them to the LLM, call one directly: the Spark
applications the History Server knows about (the notebooks' Spark Connect sessions and the SparkApplications).

In [ ]:
spark_tools = {t["name"] for t in lu.ops_tools("spark_history")} if "spark_history" in lu.ops_servers() else set()
if "list_applications" in spark_tools:
    apps = lu.ops_call("spark_history", "list_applications", {})
    print(apps[:1500] if isinstance(apps, str) else json.dumps(apps, indent=1, default=str)[:1500])
else:
    print("no list_applications tool here; the agent below picks from the tools listed above")

## 2. Ask the agent
`lu.ask_ops` gives the model (`qwen3:4b`, tool calling) every tool of both servers, prefixed with the server name, and
lets it investigate: list first, then drill down. Each step is shown below the answer, so you can check that the
answer comes from the tools.

In [ ]:
def investigate(question):
    out = lu.ask_ops(question)
    print(f"Q: {question}\n\nA: {out['answer']}\n")
    return pd.DataFrame(out["steps"])

investigate("Did the northwind_daily DAG run? Give the state of its latest run and of each task in it. "
            "If a task failed, say why from its log.")

In [ ]:
investigate("Which Spark applications ran most recently, and which of them took longest? Name the application ids.")

## What to look for
* The first answer names the latest `northwind_daily` run and its four tasks. If `check_inputs` failed, its log says
  which table is missing: run `dbt/01` (the seeds and the marts), then trigger the DAG again in the Airflow UI
  (`http://airflow.datalab.localhost:8080` on the workstation).
* The Airflow MCP server is **read-only** and logs in as its own least-privilege user: the agent can look, not
  trigger or change anything.
* Small local models sometimes skip a step or misread an id; the steps table is how you check. A bigger model
  (`ollama pull qwen3:8b`, then `lu.ask_ops(..., model="qwen3:8b")`) investigates more thoroughly.